# Build-up vs opponent press
For each team: **who helps the build-up**, **how the team builds** (starts, lanes, pass mix, shape) and **how the opponent presses it** (block height, intensity, options cut, man vs zonal, effect).

All coordinates are in the *building* team's frame (own goal x=0, attacking +x). Tune the parameters in cell 1 after reading the printed distributions.

## 1. Setup & load

In [ ]:
# ===== 1. Setup, parameters, load =====
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from scipy.optimize import linear_sum_assignment
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

CACHE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)"
             r"\project\barca_atletico_first_half\new_cache")
PLAYER_PATH = CACHE / "player_frame_table_named.parquet"
BALL_PATH = CACHE / "ball_frame_table_named.parquet"
OUT_DIR = CACHE / "buildup_analysis"; OUT_DIR.mkdir(exist_ok=True)

# ---------------- parameters (all tunable) ----------------
L, W = 105.0, 68.0
# --- what counts as a build-up phase (all coordinates in the BUILDING team's frame: own goal x=0, attacking +x)
BUILDUP_START_X = 40.0    # possession must begin with the ball at x <= this
BUILDUP_END_X   = 52.5    # phase ends when the ball crosses this line while still in possession
MIN_PHASE_SEC   = 1.5
POSS_GAP_SEC    = 4.0     # fill possession_team gaps up to this long if both sides are the same team
CARRIER_GAP_SEC = 0.6     # fill carrier gaps up to this long if the same player is on both sides
BALL_INTERP_SEC = 1.5
LOOKBACK_REGAIN_SEC = 3.0 # opponent had the ball within this window before the phase -> "regain"
# --- passes
MIN_RUN_FRAMES  = 2
MAX_PASS_GAP_SEC= 2.5
FWD_DX, PROG_DX, LONG_PASS_M = 3.0, 10.0, 25.0
# --- press measurement
SAMPLE_SEC      = 0.5
PRESS_R, CLOSE_R, CLOSING_SPEED, CLOSING_LAG_SEC = 8.0, 3.5, 2.0, 0.5
OPTION_MIN, OPTION_MAX = 5.0, 35.0    # passing options = teammates this far from the ball
MARK_R, LANE_BLOCK_R   = 3.5, 2.0     # option is "cut" if marked within MARK_R or an opponent sits on the passing lane
HIGH_PRESS_FRONT3, MID_BLOCK_FRONT3 = 65.0, 50.0   # opponent front-3 height (m from THEIR goal)
WINDOW_SAMPLES, SHADOW_MAN, SHADOW_ZONAL = 8, 0.55, 0.30
PRESSED_PHASE_SHARE = 0.30            # phase counts as "pressed" if >=30% of samples have a presser
FLIP_LANES = False                    # set True if "left/right" come out mirrored
MIN_TRACKED = 7

# ---------------- load ----------------
need = ["frame_idx","track_id","role","team","team_name","name","number","pitch_x","pitch_y","t_sec",
        "attack_dir","x_att","carrier_track_id","possession_team","is_stoppage","is_carrier"]
P  = pd.read_parquet(PLAYER_PATH, columns=need)
Bt = pd.read_parquet(BALL_PATH)
print("player rows:", len(P), "| frames:", P.frame_idx.nunique())
print("roles:", P.role.value_counts(dropna=False).to_dict())
print("\nBALL TABLE columns:", list(Bt.columns))
print(Bt.describe().T[["count","min","max","mean"]].head(25))


## 2. Prep (orientation, possession, ball)

In [ ]:
# ===== 2. Preparation: teams, orientation, frame table, ball track =====
P = P.drop_duplicates(["frame_idx", "track_id"]).copy()
for c in ["team", "possession_team", "carrier_track_id", "number"]:
    P[c] = pd.to_numeric(P[c], errors="coerce").astype("float64")
P["is_stoppage"] = P["is_stoppage"].fillna(False).astype(bool)
P["is_carrier"]  = P["is_carrier"].fillna(False).astype(bool)

FMAX = int(P.frame_idx.max())
fps  = float(round((P.frame_idx.max() - P.frame_idx.min()) / (P.t_sec.max() - P.t_sec.min()), 2))
sec  = lambda s: max(1, int(round(s * fps)))
print(f"fps = {fps} | frames 0..{FMAX} | {FMAX/fps/60:.1f} min")

rl = P["role"].astype(str).str.lower()
P["is_ref"] = rl.str.contains("ref")
P["is_gk"]  = rl.str.startswith("goal") | rl.eq("gk")
P["is_out"] = P.team.notna() & ~P.is_gk & ~P.is_ref        # outfield players only

tn = P.dropna(subset=["team"]).groupby("team")["team_name"].agg(lambda s: s.mode().iat[0])
TEAM_NAME = {int(k): v for k, v in tn.items()}
TEAMS = sorted(TEAM_NAME)
ATT_DIR = {int(k): float(np.sign(v)) for k, v in
           P.dropna(subset=["team", "attack_dir"]).groupby("team")["attack_dir"].mean().items()}
print("teams:", TEAM_NAME, "| attack direction (+1 = towards x=105):", ATT_DIR)
assert all(v in (1.0, -1.0) for v in ATT_DIR.values()) and len(set(ATT_DIR.values())) == 2, \
    "attack_dir is not constant per team (first half should be) - check the table"

def to_frame(x, y, team):
    # absolute pitch coords -> frame in which `team` attacks towards +x (own goal at x=0)
    if ATT_DIR[int(team)] == 1.0:
        return x, y
    return L - x, W - y

# sanity: my own-frame x vs the pipeline's x_att
_d = P["team"].map(ATT_DIR)
_xo = np.where(_d == 1.0, P.pitch_x, L - P.pitch_x)
print("median |x_own - x_att| (should be ~0):", float(np.nanmedian(np.abs(_xo - P.x_att))))

# ---- track info (names are unique per identity in the named cache)
def mode_or_nan(s):
    s = s.dropna()
    return s.mode().iat[0] if len(s) else np.nan
info = P.groupby("track_id").agg(name=("name", mode_or_nan), number=("number", mode_or_nan),
                                 team=("team", mode_or_nan), is_gk=("is_gk", "max"), is_ref=("is_ref", "max"))
info.index = info.index.astype(int)
info["label"] = [(f"#{int(n)} {nm}" if pd.notna(n) else str(nm)) if pd.notna(nm) else f"id{i}"
                 for i, n, nm in zip(info.index, info.number, info.name)]
TEAM_OF  = info["team"].to_dict(); LABEL = info["label"].to_dict(); GK_IDS = set(info.index[info.is_gk])

# ---- one row per frame: possession / carrier (before dropping rows without pitch coords)
fr = (P.groupby("frame_idx").agg(possession_team=("possession_team", "first"),
                                 carrier=("carrier_track_id", "first"),
                                 stop=("is_stoppage", "max")).reindex(np.arange(FMAX + 1)))
fr["stop"] = fr["stop"].fillna(False).astype(bool)

def fill_short_gaps(s, max_len):
    # fill NaN runs <= max_len only when the value before and after the gap is identical
    isn = s.isna()
    grp = (isn != isn.shift()).cumsum()
    run_len = isn.groupby(grp).transform("sum")
    f, b = s.ffill(), s.bfill()
    m = isn & (run_len <= max_len) & (f == b)
    out = s.copy(); out[m] = f[m]
    return out

car_f = fill_short_gaps(fr["carrier"].where(~fr["stop"]), sec(CARRIER_GAP_SEC))
poss0 = fr["possession_team"].where(~fr["stop"])
poss0 = poss0.fillna(car_f.map(lambda v: TEAM_OF.get(int(v), np.nan) if pd.notna(v) else np.nan))
poss_f = fill_short_gaps(poss0, sec(POSS_GAP_SEC))
print(f"possession known: raw {fr.possession_team.notna().mean():.1%} -> after gap fill {poss_f.notna().mean():.1%} | "
      f"carrier known: raw {fr.carrier.notna().mean():.1%} -> {car_f.notna().mean():.1%}")

# ---- now keep rows with pitch coordinates
P["px"] = P.pitch_x.clip(0, L); P["py"] = P.pitch_y.clip(0, W)
P = P[P.px.notna() & P.py.notna()].copy()

# ---- ball position per frame (ball table first, carrier foot as fallback)
cands_x = ["pitch_x", "ball_pitch_x", "ball_x", "x_pitch"]; cands_y = ["pitch_y", "ball_pitch_y", "ball_y", "y_pitch"]
BALL_X_COL = next((c for c in cands_x if c in Bt.columns), None)    # override here if the wrong one is picked
BALL_Y_COL = next((c for c in cands_y if c in Bt.columns), None)
bpos = pd.DataFrame(index=np.arange(FMAX + 1), columns=["x", "y"], dtype="float64")
if BALL_X_COL and BALL_Y_COL:
    bb = Bt.drop_duplicates("frame_idx").set_index("frame_idx")[[BALL_X_COL, BALL_Y_COL]]
    bb.columns = ["x", "y"]; bpos.loc[bb.index.intersection(bpos.index)] = bb.loc[bb.index.intersection(bpos.index)]
    print(f"ball coords from ball table ({BALL_X_COL}, {BALL_Y_COL}): {bpos.x.notna().mean():.1%} of frames")
cp = P[P.is_carrier].groupby("frame_idx")[["px", "py"]].first()
bpos["x"] = bpos["x"].fillna(cp["px"]); bpos["y"] = bpos["y"].fillna(cp["py"])
bpos = bpos.interpolate(limit=sec(BALL_INTERP_SEC), limit_area="inside")
bpos["x"] = bpos["x"].clip(0, L); bpos["y"] = bpos["y"].clip(0, W)
ball_x, ball_y = bpos["x"].values, bpos["y"].values
print(f"ball position available (after carrier fallback + interpolation): {bpos.x.notna().mean():.1%}")


## 3. Build-up phases

In [ ]:
# ===== 3. Detect build-up phases (possession spells that start deep and run until halfway / loss) =====
def lane_of(y):
    if y is None or np.isnan(y): return None
    lane = "left" if y < W / 3 else ("right" if y > 2 * W / 3 else "centre")
    if FLIP_LANES: lane = {"left": "right", "right": "left"}.get(lane, lane)
    return lane

key = poss_f.fillna(-1).astype(int).values
chg = np.r_[True, key[1:] != key[:-1]]
st = np.flatnonzero(chg); en = np.r_[st[1:] - 1, len(key) - 1]
car_arr = car_f.values

def first_valid(a, s, e):
    seg = a[s:e + 1]; ok = np.flatnonzero(~np.isnan(seg))
    return seg[ok[0]] if len(ok) else np.nan
def last_valid(a, s, e):
    seg = a[max(s, 0):e + 1]; ok = np.flatnonzero(~np.isnan(seg))
    return seg[ok[-1]] if len(ok) else np.nan

rows = []
for s, e in zip(st, en):
    A = key[s]
    if A < 0: continue
    bxA, byA = to_frame(ball_x[s:e + 1], ball_y[s:e + 1], A)
    ok = np.flatnonzero(~np.isnan(bxA))
    if not len(ok) or bxA[ok[0]] > BUILDUP_START_X: continue
    i0 = ok[0]; fs = s + i0
    cross = np.flatnonzero(bxA[i0:] > BUILDUP_END_X)
    if len(cross):
        fe, outcome = fs + cross[0], "progressed"
    else:
        fe = e
        nxt = key[e + 1: e + 1 + sec(3.0)]; nxt = nxt[nxt >= 0]
        outcome = "lost" if (len(nxt) and nxt[0] != A) else "stopped/unknown"
    if (fe - fs) / fps < MIN_PHASE_SEC: continue
    seg_x, seg_y = to_frame(ball_x[fs:fe + 1], ball_y[fs:fe + 1], A)
    c0 = first_valid(car_arr, fs, fs + sec(1.5)); c1 = last_valid(car_arr, fe - sec(2.0), fe)
    prev = key[max(0, s - sec(LOOKBACK_REGAIN_SEC)): s]
    if not np.isnan(c0) and int(c0) in GK_IDS: stype = "goalkeeper"
    elif ((prev >= 0) & (prev != A)).any():      stype = "regain"
    else:                                        stype = "restart/other"
    rows.append(dict(team=int(A), team_name=TEAM_NAME[int(A)], f0=int(fs), f1=int(fe), t0=fs / fps, t1=fe / fps,
                     dur=(fe - fs) / fps, outcome=outcome, start_type=stype,
                     start_x=float(seg_x[np.flatnonzero(~np.isnan(seg_x))[0]]),
                     start_y=float(seg_y[np.flatnonzero(~np.isnan(seg_y))[0]]),
                     end_x=float(np.nanmax(seg_x[-3:])) if len(seg_x) else np.nan,
                     end_y=float(np.nanmean(seg_y[-3:])), max_x=float(np.nanmax(seg_x)),
                     start_carrier=int(c0) if not np.isnan(c0) else np.nan,
                     end_carrier=int(c1) if not np.isnan(c1) else np.nan))
PH = pd.DataFrame(rows)
assert len(PH), "no build-up phases found - loosen BUILDUP_START_X / MIN_PHASE_SEC or check possession_team"
PH.insert(0, "phase_id", np.arange(len(PH)))
PH["start_lane"] = PH.start_y.map(lane_of); PH["end_lane"] = PH.end_y.map(lane_of)
PH["start_depth"] = pd.cut(PH.start_x, [-1, 16.5, 30, BUILDUP_START_X + 0.01], labels=["own box", "16.5-30 m", "30+ m"])
PH["minute"] = PH.t0 / 60

print(f"{len(PH)} build-up phases (frames {PH.f0.min()}..{PH.f1.max()})")
print(PH.groupby("team_name").agg(phases=("phase_id", "size"), per_10min=("phase_id", lambda s: len(s) / (FMAX / fps / 60) * 10),
                                  mean_dur=("dur", "mean"),
                                  progressed=("outcome", lambda s: (s == "progressed").mean()),
                                  lost=("outcome", lambda s: (s == "lost").mean())).round(2))
PH.head()


## 4. Passes & touches

In [ ]:
# ===== 4. Passes + touches inside the build-up phases =====
c = car_f.values; valid = ~np.isnan(c)
chg = np.r_[True, (c[1:] != c[:-1]) | (valid[1:] != valid[:-1])]
rdf = pd.DataFrame({"rid": np.cumsum(chg), "car": c, "f": np.arange(len(c))})[valid]
R = rdf.groupby("rid").agg(car=("car", "first"), f0=("f", "min"), f1=("f", "max")).reset_index(drop=True)
R["n"] = R.f1 - R.f0 + 1
R = R[R.n >= MIN_RUN_FRAMES].reset_index(drop=True)
R["team"] = R.car.map(lambda v: TEAM_OF.get(int(v), np.nan))

nx = R.shift(-1)
gap = nx.f0 - R.f1 - 1
ok = nx.car.notna() & (nx.car != R.car) & (gap <= sec(MAX_PASS_GAP_SEC)) & (nx.team == R.team) & R.team.notna()
PS = pd.DataFrame({"team": R.team[ok].astype(int), "passer": R.car[ok].astype(int), "receiver": nx.car[ok].astype(int),
                   "f_rel": R.f1[ok].astype(int), "f_recv": nx.f0[ok].astype(int)}).reset_index(drop=True)
for col in ["x0", "y0", "x1", "y1"]: PS[col] = np.nan
for t in TEAMS:
    m = PS.team == t
    PS.loc[m, "x0"], PS.loc[m, "y0"] = to_frame(ball_x[PS.f_rel[m].values], ball_y[PS.f_rel[m].values], t)
    PS.loc[m, "x1"], PS.loc[m, "y1"] = to_frame(ball_x[PS.f_recv[m].values], ball_y[PS.f_recv[m].values], t)
PS = PS.dropna(subset=["x0", "y0", "x1", "y1"]).reset_index(drop=True)
PS["dx"] = PS.x1 - PS.x0; PS["length"] = np.hypot(PS.dx, PS.y1 - PS.y0)
PS["direction"] = np.select([PS.dx >= FWD_DX, PS.dx <= -FWD_DX], ["forward", "backward"], "lateral")
PS["is_long"] = PS.length >= LONG_PASS_M; PS["is_prog"] = PS.dx >= PROG_DX

ph_k = PH[["phase_id", "team", "f0", "f1"]].rename(columns={"f0": "ph_f0", "f1": "ph_f1"}).sort_values("ph_f0")
def assign_phase(df, fcol):
    m = pd.merge_asof(df.sort_values(fcol), ph_k, left_on=fcol, right_on="ph_f0", by="team", direction="backward")
    return m[m[fcol] <= m.ph_f1].dropna(subset=["phase_id"]).assign(phase_id=lambda d: d.phase_id.astype(int))

PASS_PH = assign_phase(PS, "f_rel")                      # passes released during a build-up phase

TCH = R.dropna(subset=["team"]).copy(); TCH["team"] = TCH.team.astype(int); TCH["car"] = TCH.car.astype(int)
TCH["x"] = np.nan; TCH["y"] = np.nan
for t in TEAMS:
    m = TCH.team == t
    TCH.loc[m, "x"], TCH.loc[m, "y"] = to_frame(ball_x[TCH.f0[m].values], ball_y[TCH.f0[m].values], t)
TOUCH_PH = assign_phase(TCH.rename(columns={"f0": "f_touch"}).assign(f0=lambda d: d.f_touch), "f_touch")

pn = PASS_PH.groupby("phase_id").size().rename("n_passes")
PH = PH.drop(columns=["n_passes"], errors="ignore").merge(pn, on="phase_id", how="left").fillna({"n_passes": 0})
PH["n_passes"] = PH.n_passes.astype(int)
print(f"{len(PASS_PH)} passes and {len(TOUCH_PH)} touches inside {len(PH)} build-up phases")
print(PASS_PH.groupby("team").agg(passes=("passer", "size"), fwd=("direction", lambda s: (s == "forward").mean()),
                                  lateral=("direction", lambda s: (s == "lateral").mean()),
                                  back=("direction", lambda s: (s == "backward").mean()),
                                  long=("is_long", "mean"), mean_len=("length", "mean")).rename(index=TEAM_NAME).round(2))


## 5. Opponent press per phase

In [ ]:
# ===== 5. Opponent press during each build-up phase (sampled every SAMPLE_SEC) =====
def pt_seg_dist(pts, a, b):
    # distance from each point in pts (n,2) to each segment a->b[j] (m,2): returns (n,m)
    ab = b - a; ap = pts[:, None, :] - a
    t = np.clip((ap * ab[None]).sum(-1) / np.maximum((ab ** 2).sum(-1)[None], 1e-9), 0, 1)
    return np.linalg.norm(pts[:, None, :] - (a + t[..., None] * ab[None]), axis=-1)

def lines_shape(x, k=3):
    x = np.sort(x); n = len(x)
    if n < 8: return None
    cut = np.sort(np.argsort(np.diff(x))[-(k - 1):])
    return "-".join(str(len(p)) for p in np.split(np.arange(n), cut + 1))

def block_of(v):
    if pd.isna(v): return None
    return "high press" if v >= HIGH_PRESS_FRONT3 else ("mid block" if v >= MID_BLOCK_FRONT3 else "low block")

# per-player distance to ball + closing speed (m/s, positive = approaching the ball)
Pf = P[P.team.notna()].sort_values(["track_id", "frame_idx"]).copy()
Pf["d_ball"] = np.hypot(Pf.px.values - ball_x[Pf.frame_idx.values], Pf.py.values - ball_y[Pf.frame_idx.values])
lag = sec(CLOSING_LAG_SEC); gb = Pf.groupby("track_id")
Pf["closing"] = np.where((Pf.frame_idx - gb["frame_idx"].shift(lag)) == lag,
                         (gb["d_ball"].shift(lag) - Pf.d_ball) / (lag / fps), np.nan)

step = sec(SAMPLE_SEC)
S = pd.concat([pd.DataFrame({"phase_id": r.phase_id, "A": r.team, "frame_idx": np.arange(r.f0, r.f1 + 1, step)})
               for r in PH.itertuples()], ignore_index=True)
Ps = Pf.merge(S, on="frame_idx", how="inner")
print(f"{len(S)} sampled frames over {len(PH)} phases | player rows used: {len(Ps)}")

posA, posB, mrows, orows = {}, {}, [], []
for (pid, f), g in Ps.groupby(["phase_id", "frame_idx"]):
    A = int(g.A.iat[0])
    if np.isnan(ball_x[f]): continue
    bxA, byA = to_frame(ball_x[f], ball_y[f], A); bxy = np.array([bxA, byA])
    gA = g[g.team == A]; gB = g[g.team != A]; Ao = gA[gA.is_out]; Bo = gB[gB.is_out]
    if len(Ao) < 5 or len(Bo) < 5: continue
    axo, ayo = to_frame(Ao.px.values, Ao.py.values, A); bxx, byy = to_frame(Bo.px.values, Bo.py.values, A)
    Bxy = np.c_[bxx, byy]; bOwn = L - bxx                       # opponent height measured from THEIR goal
    dB, cl = Bo.d_ball.values, Bo.closing.values
    pm = (dB <= CLOSE_R) | ((dB <= PRESS_R) & (np.nan_to_num(cl, nan=-9) >= CLOSING_SPEED))
    car = car_arr[f]
    d = dict(phase_id=pid, frame_idx=f, A=A, nA=len(Ao), nB=len(Bo), ball_xA=bxA, ball_yA=byA,
             n_press=int(pm.sum()), d_nearest=float(dB.min()),
             press_own_x=float(bOwn[pm].mean()) if pm.any() else np.nan,
             mean_nearestB=float(np.linalg.norm(np.c_[axo, ayo][:, None] - Bxy[None], axis=-1).min(1).mean()),
             carrier=car)
    if len(Bo) >= MIN_TRACKED:
        d.update(front3=float(np.sort(bOwn)[-3:].mean()), b_len=float(np.percentile(bOwn, 90) - np.percentile(bOwn, 10)),
                 b_wid=float(np.percentile(byy, 90) - np.percentile(byy, 10)), b_cx=float(bOwn.mean()), shape_B=lines_shape(bOwn))
    if len(Ao) >= MIN_TRACKED + 2: d["shape_A"] = lines_shape(axo)
    # passing options of the build-up team (teammates incl. GK, not the carrier) and how many the press cuts
    Aopt = gA[gA.track_id != car]
    oxy = np.c_[to_frame(Aopt.px.values, Aopt.py.values, A)]
    dist_o = np.linalg.norm(oxy - bxy, axis=1); sel = (dist_o >= OPTION_MIN) & (dist_o <= OPTION_MAX)
    if sel.any():
        O = oxy[sel]
        cut = (np.linalg.norm(Bxy[:, None] - O[None], axis=-1).min(0) <= MARK_R) | (pt_seg_dist(Bxy, bxy, O) <= LANE_BLOCK_R).any(0)
        d.update(n_options=int(sel.sum()), opt_cut_share=float(cut.mean()), n_open=int((~cut).sum()))
        cut_full = np.zeros(len(Aopt), bool); cut_full[np.flatnonzero(sel)] = cut
    else:
        cut_full = np.zeros(len(Aopt), bool)
    for tid, s_, c_ in zip(Aopt.track_id.values, sel, cut_full):
        orows.append((pid, f, int(tid), bool(s_), bool(s_ and not c_)))
    # target of the press (who has the ball)
    if not np.isnan(car):
        d["target"] = ("GK" if int(car) in GK_IDS else
                       ("centre-back" if (bxA < 35 and lane_of(byA) == "centre") else
                        ("wide defender" if bxA < 35 else "midfield")))
    mrows.append(d)
    posA[(pid, f)] = (Ao.track_id.values.astype(int), np.c_[axo, ayo]); posB[(pid, f)] = (Bo.track_id.values.astype(int), Bxy)

M = pd.DataFrame(mrows); OPT = pd.DataFrame(orows, columns=["phase_id", "frame_idx", "track_id", "in_range", "open"])
M["block"] = M.front3.map(block_of)

# ---- man-oriented vs zonal: shadowing index (1 = opponent mirrors his man, 0 = holds position)
def shadow_vals(keys):
    ia, xa = posA[keys[0]]; ib, xb = posB[keys[0]]
    C = np.linalg.norm(xb[:, None] - xa[None], axis=-1); r, c_ = linear_sum_assignment(C); out = []
    for i, j in zip(r, c_):
        if C[i, j] > 12: continue
        ser = []
        for k in keys:
            ma = np.flatnonzero(posA[k][0] == ia[j]); mb = np.flatnonzero(posB[k][0] == ib[i])
            if len(ma) and len(mb): ser.append((posA[k][1][ma[0]], posB[k][1][mb[0]]))
        if len(ser) < 5: continue
        a = np.array([s[0] for s in ser]); b = np.array([s[1] for s in ser]); sa = np.sqrt(a.var(0).sum())
        if sa < 1.5: continue
        out.append(float(np.clip(1 - np.sqrt((b - a).var(0).sum()) / sa, -1, 1)))
    return out
sh = {}
for pid, g in M.groupby("phase_id"):
    ks = [(pid, f) for f in g.frame_idx.values if (pid, f) in posA]; vals = []
    for i in range(0, len(ks) - WINDOW_SAMPLES + 1, WINDOW_SAMPLES): vals += shadow_vals(ks[i:i + WINDOW_SAMPLES])
    if vals: sh[pid] = (np.median(vals), len(vals))
SH = pd.DataFrame(sh, index=["shadow_index", "n_pairs"]).T

modeS = lambda s: s.dropna().mode().iat[0] if s.notna().any() else None
agg = M.groupby("phase_id").agg(n_samp=("frame_idx", "size"), press_share=("n_press", lambda s: (s > 0).mean()),
                                n_press_mean=("n_press", "mean"), n_press_max=("n_press", "max"),
                                d_nearest=("d_nearest", "median"), front3=("front3", "mean"), b_len=("b_len", "mean"),
                                b_wid=("b_wid", "mean"), opt_cut=("opt_cut_share", "mean"), tight=("mean_nearestB", "mean"),
                                shape_A=("shape_A", modeS), shape_B=("shape_B", modeS), first_f=("frame_idx", "min"))
fp = M[M.n_press > 0].groupby("phase_id").frame_idx.min().rename("first_press_f")
PH = PH.drop(columns=[c for c in PH.columns if c in list(agg.columns) + ["first_press_f", "shadow_index", "n_pairs"]], errors="ignore")
PH = PH.merge(agg.drop(columns="first_f"), on="phase_id", how="left").merge(fp, on="phase_id", how="left").merge(SH, left_on="phase_id", right_index=True, how="left")
PH["t_to_press"] = (PH.first_press_f - PH.f0) / fps
PH["press_block"] = PH.front3.map(block_of)
PH["pressed"] = PH.press_share >= PRESSED_PHASE_SHARE
PH["press_style"] = np.select([PH.shadow_index >= SHADOW_MAN, PH.shadow_index <= SHADOW_ZONAL, PH.shadow_index.notna()],
                              ["man-oriented", "zonal", "mixed"], None)

# under-pressure flag per pass
Mk = M[["phase_id", "frame_idx", "n_press"]].sort_values("frame_idx")
PASS_PH = pd.merge_asof(PASS_PH.sort_values("f_rel"), Mk, left_on="f_rel", right_on="frame_idx", by="phase_id",
                        direction="nearest", tolerance=step).rename(columns={"n_press": "press_at_release"})
PASS_PH["under_press"] = PASS_PH.press_at_release.fillna(0) >= 1

print("front-3 height distribution (use it to tune HIGH_PRESS_FRONT3 / MID_BLOCK_FRONT3):")
print(M.front3.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_dict())
print("shadow index distribution (tune SHADOW_MAN / SHADOW_ZONAL):", PH.shadow_index.describe(percentiles=[.25, .5, .75]).round(2).to_dict())
print(PH.groupby("team_name").agg(pressed_share=("pressed", "mean"), press_mean=("n_press_mean", "mean"),
                                  front3=("front3", "mean"), opt_cut=("opt_cut", "mean"), t_to_press=("t_to_press", "median")).round(2))


## 6. Who helps the build-up

In [ ]:
# ===== 6. Who helps the build-up? (per team) =====
def involvement(team):
    ph = PH[PH.team == team]; ps = PASS_PH[PASS_PH.team == team]; tc = TOUCH_PH[TOUCH_PH.team == team]
    op = OPT[OPT.phase_id.isin(ph.phase_id)]
    sup = (op.groupby("track_id").agg(samples=("in_range", "size"), in_range=("in_range", "sum"), open=("open", "sum")))
    rows = []
    for tid in sorted(set(ps.passer) | set(ps.receiver) | set(tc.car)):
        made, recv, tch = ps[ps.passer == tid], ps[ps.receiver == tid], tc[tc.car == tid]
        inv = set(tch.phase_id) | set(made.phase_id) | set(recv.phase_id)
        rows.append(dict(player=LABEL.get(int(tid), tid), track_id=int(tid), is_gk=int(tid) in GK_IDS,
                         phases_involved=len(inv), involvement=len(inv) / len(ph), touches=len(tch),
                         passes_made=len(made), passes_received=len(recv),
                         progressive_passes=int(made.is_prog.sum()), long_passes=int(made.is_long.sum()),
                         started=int((ph.start_carrier == tid).sum()),
                         exits=int(((ph.end_carrier == tid) & (ph.outcome == "progressed")).sum()),
                         lost_ball=int(((ph.end_carrier == tid) & (ph.outcome == "lost")).sum()),
                         passes_under_press=int(made.under_press.sum()),
                         option_share=(sup.loc[tid, "open"] / sup.loc[tid, "samples"]) if tid in sup.index else np.nan,
                         avg_x=tch.x.mean(), avg_y=tch.y.mean()))
    T = pd.DataFrame(rows)
    T["link_total"] = T.passes_made + T.passes_received
    T["forward_work"] = T.progressive_passes + T.exits
    crit = ["phases_involved", "link_total", "forward_work", "option_share"]
    T["help_score"] = T[crit].rank(pct=True).mean(axis=1)           # mean percentile rank over the 4 criteria
    return T.sort_values("help_score", ascending=False).reset_index(drop=True)

INV = {t: involvement(t) for t in TEAMS}
for t in TEAMS:
    print(f"\n=== {TEAM_NAME[t]} - players ranked by build-up contribution ({(PH.team == t).sum()} phases) ===")
    print(INV[t][["player", "help_score", "phases_involved", "involvement", "touches", "passes_made", "passes_received",
                  "progressive_passes", "exits", "started", "option_share", "avg_x", "avg_y"]].head(12).round(2).to_string(index=False))


## 7. Strategy summary & read-out

In [ ]:
# ===== 7. Build-up strategy per team + how the opponent presses it =====
def share(s, v): return float((s == v).mean()) if len(s) else np.nan
def top(s, k=3):
    vc = s.dropna().value_counts(normalize=True).head(k)
    return ", ".join(f"{i} {v:.0%}" for i, v in vc.items())

rows = []
for t in TEAMS:
    ph, ps, mm = PH[PH.team == t], PASS_PH[PASS_PH.team == t], M.merge(PH[["phase_id", "team"]], on="phase_id").query("team == @t")
    opp = [x for x in TEAMS if x != t][0]
    gk_phases = ph.phase_id.isin(TOUCH_PH[(TOUCH_PH.team == t) & TOUCH_PH.car.isin(GK_IDS)].phase_id)
    pressed_samples = mm[mm.n_press > 0]
    rows.append({
        "team": TEAM_NAME[t], "opponent": TEAM_NAME[opp], "phases": len(ph),
        # --- how the team builds
        "start via GK": share(ph.start_type, "goalkeeper"), "start via regain": share(ph.start_type, "regain"),
        "GK touches the ball": float(gk_phases.mean()),
        "progressed": share(ph.outcome, "progressed"), "lost": share(ph.outcome, "lost"),
        "passes / phase": ph.n_passes.mean(), "phase duration (s)": ph.dur.mean(),
        "fwd %": share(ps.direction, "forward"), "lateral %": share(ps.direction, "lateral"), "back %": share(ps.direction, "backward"),
        "long %": float(ps.is_long.mean()), "avg pass (m)": ps.length.mean(),
        "exit lanes": top(ph[ph.outcome == "progressed"].end_lane), "start lanes": top(ph.start_lane),
        "usual shape (def-mid-att)": modeS(ph.shape_A),
        # --- how the opponent presses it
        "opp press block": top(PH[PH.team == t].press_block), "opp shape": modeS(ph.shape_B),
        "opp front-3 height (m)": ph.front3.mean(), "pressed phases": float(ph.pressed.mean()),
        "pressers (avg when pressed)": pressed_samples.n_press.mean(), "options cut %": ph.opt_cut.mean(),
        "time to first press (s)": ph.t_to_press.median(), "press style": top(ph.press_style),
        "press target": top(pressed_samples.target),
        # --- how the team reacts
        "progress | pressed": share(ph[ph.pressed].outcome, "progressed"), "progress | free": share(ph[~ph.pressed].outcome, "progressed"),
        "lost | pressed": share(ph[ph.pressed].outcome, "lost"), "lost | free": share(ph[~ph.pressed].outcome, "lost"),
        "long % | pressed": float(ps[ps.under_press].is_long.mean()) if ps.under_press.any() else np.nan,
        "long % | free": float(ps[~ps.under_press].is_long.mean()) if (~ps.under_press).any() else np.nan,
        "fwd % | pressed": share(ps[ps.under_press].direction, "forward"), "fwd % | free": share(ps[~ps.under_press].direction, "forward"),
    })
SUM = pd.DataFrame(rows).set_index("team").T
display(SUM.map(lambda v: round(v, 2) if isinstance(v, float) else v)) if "display" in globals() else print(SUM)

# ---- plain-language read-out
def narrative(t):
    r = SUM[TEAM_NAME[t]]; I = INV[t]; ph = PH[PH.team == t]
    p = lambda k: "n/a" if pd.isna(r[k]) else f"{r[k]:.0%}"
    best = I.head(3).player.tolist(); prog = I.sort_values("forward_work", ascending=False).head(3).player.tolist()
    outlets = I[(~I.is_gk)].sort_values("option_share", ascending=False).head(2).player.tolist()
    lines = [f"{TEAM_NAME[t]} BUILD-UP ({int(r['phases'])} phases)",
      f"- Starts: {p('start via GK')} from the goalkeeper, {p('start via regain')} after winning the ball; start lanes {r['start lanes']}.",
      f"- Style: {r['passes / phase']:.1f} passes per phase over {r['phase duration (s)']:.0f}s; passes are {p('fwd %')} forward / {p('lateral %')} lateral / {p('back %')} backward, "
      f"{p('long %')} long; GK involved in {p('GK touches the ball')} of phases.",
      f"- Result: {p('progressed')} reach halfway in possession, {p('lost')} are lost in their own half; exits go through {r['exit lanes']}.",
      f"- Shape in build-up (back-mid-front lines): {r['usual shape (def-mid-att)']}.",
      f"- Most helpful: {', '.join(best)}. Moves it forward: {', '.join(prog)}. Most available as an option: {', '.join(outlets)}.",
      f"{TEAM_NAME[[x for x in TEAMS if x != t][0]]} PRESS vs {TEAM_NAME[t]}",
      f"- Block: {r['opp press block']} (front-3 on average {r['opp front-3 height (m)']:.0f} m from their goal), shape {r['opp shape']}.",
      f"- Intensity: {p('pressed phases')} of phases are pressed, {r['pressers (avg when pressed)']:.1f} pressers on average, first pressure after {r['time to first press (s)']:.1f}s; "
      f"{p('options cut %')} of passing options are marked or lane-blocked.",
      f"- Marking: {r['press style']}; pressure mostly lands on: {r['press target']}.",
      f"- Effect: progression {p('progress | free')} when free vs {p('progress | pressed')} when pressed; losses {p('lost | free')} vs {p('lost | pressed')}; "
      f"long passes {p('long % | free')} vs {p('long % | pressed')}."]
    return "\n".join(lines)
for t in TEAMS: print(narrative(t), "\n")


## 8. Visuals

In [ ]:
# ===== 8. Visuals (one block per team) =====
COL = {TEAMS[0]: "#1f5fbf", TEAMS[1]: "#c0392b"}
def draw_pitch(ax, lw=1.1, c="#444"):
    ax.set_xlim(-2, L + 2); ax.set_ylim(W + 2, -2); ax.set_aspect("equal"); ax.axis("off")   # y down: larger y = right-hand side when attacking >
    ax.add_patch(Rectangle((0, 0), L, W, fill=False, ec=c, lw=lw)); ax.plot([L / 2] * 2, [0, W], c=c, lw=lw)
    ax.add_patch(Circle((L / 2, W / 2), 9.15, fill=False, ec=c, lw=lw))
    for x0, w in [(0, 16.5), (L - 16.5, 16.5)]: ax.add_patch(Rectangle((x0, W / 2 - 20.16), w, 40.32, fill=False, ec=c, lw=lw))
    for x0, w in [(0, 5.5), (L - 5.5, 5.5)]: ax.add_patch(Rectangle((x0, W / 2 - 9.16), w, 18.32, fill=False, ec=c, lw=lw))
    ax.axvline(BUILDUP_START_X, c="#999", lw=.8, ls=":"); ax.axvline(BUILDUP_END_X, c="#999", lw=.8, ls=":")

OUTC = {"progressed": "#2e8b57", "lost": "#c0392b", "stopped/unknown": "#888"}
for t in TEAMS:
    ph, ps, I = PH[PH.team == t], PASS_PH[PASS_PH.team == t], INV[t]; name = TEAM_NAME[t]
    # ---------- Figure A: how the team builds ----------
    fig, ax = plt.subplots(1, 3, figsize=(24, 6.4), gridspec_kw={"width_ratios": [1.25, 1.25, 1]})
    a = ax[0]; draw_pitch(a); a.set_title(f"{name}: passing network in build-up (node = avg touch position)")
    pos = I.dropna(subset=["avg_x"]).set_index("track_id")
    E = ps.groupby(["passer", "receiver"]).size().reset_index(name="n"); E["a"] = E[["passer", "receiver"]].min(1); E["b"] = E[["passer", "receiver"]].max(1)
    E = E.groupby(["a", "b"]).n.sum().reset_index(); E = E[E.n >= max(2, 0.03 * len(ps))]
    for r in E.itertuples():
        if r.a in pos.index and r.b in pos.index:
            a.plot([pos.at[r.a, "avg_x"], pos.at[r.b, "avg_x"]], [pos.at[r.a, "avg_y"], pos.at[r.b, "avg_y"]], c=COL[t], alpha=.35, lw=.6 + 5 * r.n / E.n.max(), zorder=2)
    a.scatter(pos.avg_x, pos.avg_y, s=60 + 900 * pos.touches / pos.touches.max(), c=COL[t], ec="white", zorder=3)
    for tid, r in pos.iterrows(): a.text(r.avg_x, r.avg_y - 3.2, r.player, ha="center", fontsize=8, zorder=4)
    b = ax[1]; draw_pitch(b); b.set_title(f"{name}: where phases start -> end (green = reached halfway, red = lost)")
    for r in ph.itertuples():
        b.annotate("", xy=(r.end_x, r.end_y), xytext=(r.start_x, r.start_y),
                   arrowprops=dict(arrowstyle="->", color=OUTC[r.outcome], alpha=.45, lw=1.1))
    b.scatter(ph.start_x, ph.start_y, s=12, c="k", zorder=3)
    c_ = ax[2]; top_ = I.head(10).iloc[::-1]
    c_.barh(top_.player, top_.involvement, color=[("#999" if g else COL[t]) for g in top_.is_gk])
    for y_, (inv, ms, pr, ex) in enumerate(zip(top_.involvement, top_.passes_made, top_.progressive_passes, top_.exits)):
        c_.text(inv + .01, y_, f"made {ms} | prog {pr} | exits {ex}", va="center", fontsize=8)
    c_.set_xlim(0, 1.35); c_.set_xlabel("share of build-up phases the player is involved in"); c_.set_title(f"{name}: most helpful players (grey = GK)")
    plt.tight_layout(); plt.show()

    # ---------- Figure B: the opponent press against this build-up ----------
    mm = M.merge(PH[["phase_id", "team"]], on="phase_id").query("team == @t")
    fig, ax = plt.subplots(1, 3, figsize=(24, 6.2), gridspec_kw={"width_ratios": [1.25, 1.25, 1]})
    a = ax[0]; draw_pitch(a); a.set_title(f"{name} (blue) vs opponent press (red): average positions, pressed moments")
    pk = [k for k in posA if k[0] in set(ph.phase_id) and mm.set_index(["phase_id", "frame_idx"]).n_press.get(k, 0) >= 1]
    for D, colr in [(posA, COL[t]), (posB, "#e67e22")]:
        acc = {}
        for k in pk:
            for tid, xy in zip(*D[k]): acc.setdefault(tid, []).append(xy)
        for tid, v in acc.items():
            if len(v) >= 5:
                m_ = np.mean(v, 0); a.scatter(*m_, s=170, c=colr, ec="white", zorder=3); a.text(m_[0], m_[1] - 3, LABEL.get(tid, tid) if D is posA else "", ha="center", fontsize=7)
    b = ax[1]; draw_pitch(b); b.set_title("where the build-up team is pressed (ball location, darker = more pressers)")
    pr = mm[mm.n_press > 0]
    hb = b.hexbin(pr.ball_xA, pr.ball_yA, C=pr.n_press, reduce_C_function=np.mean, gridsize=14, extent=(0, L, 0, W), cmap="Reds", mincnt=2, alpha=.85)
    b.set_xlim(-2, L + 2); b.set_ylim(W + 2, -2)
    c_ = ax[2]
    vc = ph.press_block.value_counts(normalize=True).reindex(["high press", "mid block", "low block"]).fillna(0)
    c_.bar(vc.index, vc.values, color="#e67e22"); c_.set_ylim(0, 1); c_.set_title(f"opponent block height vs {name} build-up")
    for i, v in enumerate(vc.values): c_.text(i, v + .02, f"{v:.0%}", ha="center")
    plt.tight_layout(); plt.show()

    # ---------- Figure C: effect of the press ----------
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
    oc = ph.groupby("pressed").outcome.value_counts(normalize=True).unstack().reindex([False, True]).fillna(0)
    oc.index = ["free (not pressed)", "pressed"]
    oc.reindex(columns=["progressed", "lost", "stopped/unknown"]).plot.bar(ax=ax[0], color=[OUTC[k] for k in ["progressed", "lost", "stopped/unknown"]], rot=0)
    ax[0].set_title(f"{name}: build-up outcome, pressed vs free"); ax[0].set_ylim(0, 1)
    mix = ps.groupby("under_press").direction.value_counts(normalize=True).unstack().reindex([False, True]).fillna(0)
    mix.index = ["no pressure at release", "pressed at release"]; mix["long"] = ps.groupby("under_press").is_long.mean().reindex([False, True]).values
    mix.plot.bar(ax=ax[1], rot=0); ax[1].set_title(f"{name}: pass mix, pressed vs free"); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

# ---------- side-by-side: pressure-vs-time (time to press / pressers per phase) ----------
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for t in TEAMS:
    d = PH[PH.team == t]
    ax[0].hist(d.front3.dropna(), bins=15, alpha=.55, color=COL[t], label=f"press vs {TEAM_NAME[t]} build-up")
    ax[1].hist(d.n_press_mean.dropna(), bins=15, alpha=.55, color=COL[t], label=TEAM_NAME[t] + " building")
ax[0].axvline(HIGH_PRESS_FRONT3, c="k", ls=":"); ax[0].axvline(MID_BLOCK_FRONT3, c="k", ls=":")
ax[0].set_title("opponent front-3 height per phase (m from their goal)"); ax[1].set_title("average number of pressers per phase")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()


## 9. Export

In [ ]:
# ===== 9. Export =====
PH.to_csv(OUT_DIR / "buildup_phases.csv", index=False)
PASS_PH.to_csv(OUT_DIR / "buildup_passes.csv", index=False)
M.to_csv(OUT_DIR / "buildup_press_samples.csv", index=False)
for t in TEAMS: INV[t].to_csv(OUT_DIR / f"buildup_players_{TEAM_NAME[t]}.csv", index=False)
SUM.to_csv(OUT_DIR / "buildup_team_summary.csv")
print("saved to", OUT_DIR)
